<a href="https://colab.research.google.com/github/siddhimishra20/compiler-construction-lab/blob/main/week1/compi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Requirements

In [ ]:
# 1. Install flex and bison
!apt-get install -y flex bison

## WEEK 1 Parsing



In [ ]:
%%writefile compi.l
%{
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include "compi.tab.h"

int line = 1;
%}

DIGIT      [0-9]
LETTER     [A-Za-z_]
ID         {LETTER}({LETTER}|{DIGIT})*
ICONST     0|[1-9]{DIGIT}*
FCONST     {DIGIT}+"."{DIGIT}+

%%

"int"        { return INT; }
"float"      { return FLOAT; }
"if"         { return IF; }
"else"       { return ELSE; }
"while"      { return WHILE; }
"print"      { return PRINT; }

{FCONST}     { yylval.fval = atof(yytext); return FCONST; }
{ICONST}     { yylval.ival = atoi(yytext); return ICONST; }
{ID}         { yylval.sval = strdup(yytext); return ID; }

"=="         { return EQ; }
"!="         { return NE; }
"<="         { return LE; }
">="         { return GE; }
"&&"         { return AND; }
"||"         { return OR; }

"<"          { return '<'; }
">"          { return '>'; }
"="          { return '='; }

"+"          { return '+'; }
"-"          { return '-'; }
"*"          { return '*'; }
"/"          { return '/'; }
"%"          { return '%'; }
"!"          { return '!'; }

"("          { return '('; }
")"          { return ')'; }
"{"          { return '{'; }
"}"          { return '}'; }
";"          { return ';'; }
","          { return ','; }

[ \t]+       { }
\n           { line++; }

.            { printf("Lexical error at line %d: %s\n", line, yytext); }

%%

int yywrap() { return 1; }

Overwriting compi.l


## YACC(Token Table)

In [ ]:
%%writefile compi.y
%{
#include <stdio.h>
#include <stdlib.h>

extern int line;
int yylex(void);
void yyerror(char *s);
%}

%union {
    int ival;
    float fval;
    char* sval;
}

%token <sval> ID
%token <ival> ICONST
%token <fval> FCONST
%token INT FLOAT IF ELSE WHILE PRINT
%token EQ NE LE GE AND OR

%%

program:
    program token
    | /* empty */
    ;

token:
      INT      { printf("%-8d %-15s %s\n", line, "KEYWORD", "int"); }
    | FLOAT    { printf("%-8d %-15s %s\n", line, "KEYWORD", "float"); }
    | IF       { printf("%-8d %-15s %s\n", line, "KEYWORD", "if"); }
    | ELSE     { printf("%-8d %-15s %s\n", line, "KEYWORD", "else"); }
    | WHILE    { printf("%-8d %-15s %s\n", line, "KEYWORD", "while"); }
    | PRINT    { printf("%-8d %-15s %s\n", line, "KEYWORD", "print"); }

    | ID       { printf("%-8d %-15s %s\n", line, "IDENTIFIER", $1); free($1); }

    | ICONST   { printf("%-8d %-15s %d\n", line, "INTEGER", $1); }

    | FCONST   { printf("%-8d %-15s %.2f\n", line, "FLOAT", $1); }

    | EQ       { printf("%-8d %-15s %s\n", line, "OPERATOR", "=="); }
    | NE       { printf("%-8d %-15s %s\n", line, "OPERATOR", "!="); }
    | LE       { printf("%-8d %-15s %s\n", line, "OPERATOR", "<="); }
    | GE       { printf("%-8d %-15s %s\n", line, "OPERATOR", ">="); }
    | AND      { printf("%-8d %-15s %s\n", line, "OPERATOR", "&&"); }
    | OR       { printf("%-8d %-15s %s\n", line, "OPERATOR", "||"); }

    | '='      { printf("%-8d %-15s %s\n", line, "ASSIGNMENT", "="); }

    | '+'      { printf("%-8d %-15s %s\n", line, "ARITHMETIC_OP", "+"); }
    | '-'      { printf("%-8d %-15s %s\n", line, "ARITHMETIC_OP", "-"); }
    | '*'      { printf("%-8d %-15s %s\n", line, "ARITHMETIC_OP", "*"); }
    | '/'      { printf("%-8d %-15s %s\n", line, "ARITHMETIC_OP", "/"); }
    | '%'      { printf("%-8d %-15s %s\n", line, "ARITHMETIC_OP", "%"); }

    | '!'      { printf("%-8d %-15s %s\n", line, "LOGICAL_OP", "!"); }

    | '<'      { printf("%-8d %-15s %s\n", line, "RELATIONAL_OP", "<"); }
    | '>'      { printf("%-8d %-15s %s\n", line, "RELATIONAL_OP", ">"); }

    | '('      { printf("%-8d %-15s %s\n", line, "DELIMITER", "("); }
    | ')'      { printf("%-8d %-15s %s\n", line, "DELIMITER", ")"); }
    | '{'      { printf("%-8d %-15s %s\n", line, "DELIMITER", "{"); }
    | '}'      { printf("%-8d %-15s %s\n", line, "DELIMITER", "}"); }

    | ';'      { printf("%-8d %-15s %s\n", line, "DELIMITER", ";"); }
    | ','      { printf("%-8d %-15s %s\n", line, "DELIMITER", ","); }
    ;

%%

int main() {

    printf("\nTOKEN TABLE\n");
    printf("---------------------------------------------\n");
    printf("%-8s %-15s %s\n", "LINE", "TOKEN TYPE", "LEXEME");
    printf("---------------------------------------------\n");

    yyparse();
    return 0;
}

void yyerror(char *s) {
    fprintf(stderr, "Parse error at line %d: %s\n", line, s);
}

Overwriting compi.y


## Input Program

In [ ]:
%%writefile program.txt
int a;
int b;
int sum;
float avg;
a = 2 * (3 + 4);
b = 15;
sum = 0;
while (a < b && b != 0) {
 int temp;
 temp = a * 2;
 if ((temp % 3 == 0) || (a > 5)) {
 sum = sum + temp;
 } else {
 sum = sum - 1;
 }
 a = a + 1;
}
avg = sum / (b - a);
if (!(avg < 5.0)) {
 print(sum);
} else {
 print(avg);
}



Overwriting program.txt


## Token Stream

In [ ]:
# 2. Generate Parser and Lexer files
!bison -d compi.y
!flex compi.l

# 3. Compile the C files
!gcc compi.tab.c lex.yy.c -o compiler -lfl

# 4. Run the program
!./compiler < program.txt

Line 1: Syntactic Validation [Declaration: a]
Line 2: Syntactic Validation [Declaration: b]
Line 3: Syntactic Validation [Declaration: sum]
Line 4: Syntactic Validation [Declaration: avg]
Line 5: Syntactic Validation [Assignment to a]
Line 6: Syntactic Validation [Assignment to b]
Line 7: Syntactic Validation [Assignment to sum]
Line 9: Syntactic Validation [Declaration: temp]
Line 10: Syntactic Validation [Assignment to temp]
Line 12: Syntactic Validation [Assignment to sum]
Line 14: Syntactic Validation [Assignment to sum]
Line 15: Syntactic Validation [If-Else Block]
Line 16: Syntactic Validation [Assignment to a]
Line 17: Syntactic Validation [While Loop]
Line 18: Syntactic Validation [Assignment to avg]
Line 20: Syntactic Validation [Print Statement]
Line 22: Syntactic Validation [Print Statement]
Line 23: Syntactic Validation [If-Else Block]

RESULT: Syntactic Validation Successful.
